# 21.5 Dask:用任务图把 Python 扩展到多核/集群 / Dask: Scaling Python with Task Graphs

**中文**:Spark 很强,但它是 **JVM 世界**的——你得学它的 API、忍受 Python↔JVM 的序列化开销,还常常为几 GB 数据启动整个集群。**Dask** 是另一条路:**纯 Python 的并行计算框架**,它的口号是"**扩展你已经在用的 Python**"——`dask.dataframe` 几乎就是 pandas、`dask.array` 几乎就是 numpy,但能**并行跑在多核甚至多机上**。它的核心机制优雅得惊人:把你的计算拆成一张**任务图(task graph)**——一个由 Python 函数节点组成的 DAG——然后用调度器**把互相独立的任务并行执行**。本节从零实现一个 mini-Dask:`delayed` 惰性封装 + 任务图 + 并行调度器,并演示 `dask.dataframe` 的"分区 pandas"思想。理解了任务图,你就理解了 Dask、也更懂所有惰性计算引擎的共同套路。
**English**: Spark is powerful, but it lives in the **JVM world** — you must learn its API, tolerate Python↔JVM serialization overhead, and often spin up a whole cluster for a few GB. **Dask** is another path: a **pure-Python parallel computing framework** whose motto is "**scale the Python you already use**" — `dask.dataframe` is almost pandas, `dask.array` is almost numpy, but they **run in parallel across cores and even machines**. Its core mechanism is remarkably elegant: break your computation into a **task graph** — a DAG of Python-function nodes — then use a scheduler to **execute independent tasks in parallel**. This section builds a mini-Dask from scratch: `delayed` lazy wrapping + task graph + parallel scheduler, and demonstrates `dask.dataframe`'s "partitioned pandas" idea. Understand the task graph and you understand Dask — and better grasp the common pattern behind all lazy computation engines.

---

**中文**:**Dask 的三个核心抽象**:
**English**: **Dask's three core abstractions**:
- **中文**:**`dask.delayed`**:把任意 Python 函数**惰性化**——调用它不立即执行,而是返回一个"待办任务"节点,记录"要调用什么函数、依赖哪些其他任务"。这样一串调用就**自动织成一张任务图(DAG)**。
  **`dask.delayed`**: makes any Python function **lazy** — calling it doesn't execute immediately but returns a "to-do task" node recording "what function to call, which other tasks it depends on." A chain of calls thus **automatically weaves a task graph (DAG)**.
- **中文**:**`dask.dataframe` / `dask.array`**:把一个大 DataFrame/数组切成很多**分区(每个分区是一个普通 pandas/numpy 块)**,对它的操作被翻译成"在每个分区上跑 + 合并结果"的任务图。API 和 pandas/numpy 几乎一样。
  **`dask.dataframe` / `dask.array`**: split a big DataFrame/array into many **partitions (each a regular pandas/numpy chunk)**, and operations are translated into a task graph of "run on each partition + combine." The API is almost identical to pandas/numpy.
- **中文**:**调度器(scheduler)**:拿到任务图后,**并行执行没有依赖关系的任务**(多线程/多进程/分布式集群),自动处理依赖顺序。惰性到 `.compute()` 才真正触发(和 Spark 一样)。
  **Scheduler**: given the task graph, **executes independent tasks in parallel** (threads/processes/distributed cluster), handling dependency order automatically. Lazy until `.compute()` triggers it (like Spark).

> 💡 **面试速查 / Interview cheat-sheet（★★ Python 数据工程常考）**
> **中文**:**Dask**=纯 Python 并行计算, "扩展你已在用的 pandas/numpy/sklearn"。**三抽象**:`delayed`(惰性化任意函数→任务图)、`dask.dataframe/array`(分区的 pandas/numpy)、scheduler(并行执行独立任务, `.compute()` 触发)。**核心=任务图(DAG)**:惰性构建 → 调度器并行跑独立分支。**vs Spark**:Dask 纯 Python(无 JVM 序列化开销、能跑任意 Python 代码/科学计算栈、调试友好)、更轻量; Spark 生态更成熟、超大规模/SQL 更强、容错更硬。**vs Polars/DuckDB**:后者单机内单引擎(C++/Rust)极快, Dask 强在**并行化任意 Python 逻辑 + 多机扩展 + 无缝接 numpy/sklearn**。**何时用 Dask**:已有 pandas/numpy 代码想扩到多核/多机、或数据略超内存(out-of-core)、或做自定义并行工作流。**坑**:小数据上任务图调度开销 > 收益(不如纯 pandas); shuffle 类操作(groupby/join 跨分区)仍慢; 别对已能单机的活上 Dask。生态:`dask.distributed`(集群+dashboard)、`dask-ml`。面试金句:*"Dask 用惰性任务图把 Python 扩展到多核/多机——delayed 织出 DAG、调度器并行跑独立任务、dask.dataframe 是分区 pandas; 相比 Spark 无 JVM 开销、能跑任意 Python, 适合扩展现有 pandas/numpy 代码; 但小数据别用(调度开销), 单机极速用 Polars/DuckDB。"*
> **English**: **Dask** = pure-Python parallel computing, "scale the pandas/numpy/sklearn you already use." **Three abstractions**: `delayed` (make any function lazy → task graph), `dask.dataframe/array` (partitioned pandas/numpy), scheduler (execute independent tasks in parallel, `.compute()` triggers). **Core = the task graph (DAG)**: built lazily → scheduler runs independent branches in parallel. **vs Spark**: Dask is pure Python (no JVM serialization overhead, runs arbitrary Python / the scientific stack, debug-friendly), lighter; Spark's ecosystem is more mature, stronger at huge scale/SQL, tougher fault tolerance. **vs Polars/DuckDB**: those are single-machine single-engine (C++/Rust) and blazing fast; Dask shines at **parallelizing arbitrary Python logic + multi-machine scaling + seamless numpy/sklearn**. **When to use Dask**: existing pandas/numpy code to scale to cores/machines, data slightly over memory (out-of-core), or custom parallel workflows. **Pitfalls**: on small data the task-graph scheduling overhead > benefit (plain pandas wins); shuffle-type ops (cross-partition groupby/join) are still slow; don't Dask what already fits single-machine. Ecosystem: `dask.distributed` (cluster + dashboard), `dask-ml`. Interview line: *"Dask scales Python to cores/machines via a lazy task graph — delayed weaves a DAG, the scheduler runs independent tasks in parallel, dask.dataframe is partitioned pandas; versus Spark it has no JVM overhead and runs arbitrary Python, ideal for scaling existing pandas/numpy code; but don't use it on small data (scheduling overhead), and for single-machine speed use Polars/DuckDB."*


In [ ]:

# ============================================================
# 从零实现 mini-Dask:delayed 惰性任务图 + 并行调度器 / mini-Dask: delayed task graph + parallel scheduler
# 中文:delayed 把函数变惰性——调用只记录"待办任务"和依赖, 织成 DAG。compute() 时调度器把互相独立的
#      分支并行跑(多线程)。这正是 Dask 的核心机制。
# English: delayed makes functions lazy — calling records a "to-do task" and its deps, weaving a DAG. On compute()
#      the scheduler runs independent branches in parallel (threads). This IS Dask's core mechanism.
# ============================================================
import time
from concurrent.futures import ThreadPoolExecutor
_id=[0]
class Delayed:
    def __init__(self,func,args,name): self.func=func; self.args=args; self.name=name
    def compute(self,parallel=True): return _run(self,{},parallel)     # 触发计算 / trigger; memo caches results
    def graph_edges(self,edges=None):                                   # 收集 DAG 的边(用于画图)/ collect DAG edges
        edges=edges if edges is not None else []
        for a in self.args:
            if isinstance(a,Delayed): edges.append((a.name,self.name)); a.graph_edges(edges)
        return edges
def delayed(func):                                    # 装饰器:把普通函数变成惰性任务工厂 / lazy task factory
    def wrap(*args):
        _id[0]+=1; return Delayed(func,args,f"{func.__name__}#{_id[0]}")
    return wrap
def _run(node,memo,parallel):
    if node.name in memo: return memo[node.name]                       # 已算过就复用 / reuse computed
    deps=[a for a in node.args if isinstance(a,Delayed)]
    if parallel and len(deps)>1:                                       # 多个独立依赖 → 并行执行 / run independent deps in parallel
        with ThreadPoolExecutor(max_workers=len(deps)) as ex:
            list(ex.map(lambda n:_run(n,memo,parallel), deps))
    resolved=[_run(a,memo,parallel) if isinstance(a,Delayed) else a for a in node.args]
    memo[node.name]=node.func(*resolved)                              # 依赖就绪后执行本节点 / run this node
    return memo[node.name]

@delayed
def load(n):  time.sleep(0.2); return list(range(n))                  # 模拟耗时的加载 / simulated slow load
@delayed
def square(lst): time.sleep(0.2); return [v*v for v in lst]          # 模拟耗时的处理 / slow transform
@delayed
def total(lst):  return sum(lst)
@delayed
def combine(*parts): return sum(parts)

branches=[total(square(load(100+i))) for i in range(3)]              # 3 条独立分支 → 可并行 / 3 independent branches
result=combine(*branches)                                            # 汇总 / merge
print("构建了任务图(还没计算)。DAG 的边 / task graph built (not computed). DAG edges:")
for a,b in result.graph_edges(): print(f"   {a:>12}  →  {b}")
t=time.time(); val_par=result.compute(parallel=True);  t_par=time.time()-t
t=time.time(); val_seq=result.compute(parallel=False); t_seq=time.time()-t
print(f"\n结果 result: {val_par} (== {val_seq})")
print(f"并行 parallel: {t_par:.2f}s   串行 sequential: {t_seq:.2f}s   加速 speedup: {t_seq/t_par:.1f}x (3条独立分支)")


In [ ]:

# ============================================================
# dask.dataframe 的思想:分区 pandas + 每分区计算 + 合并 / partitioned pandas: per-partition compute + combine
# 中文:一个"大" DataFrame 切成若干分区(每个是普通 pandas 块)。groupby-求和 = 每分区各自 groupby 再合并,
#      结果与单机 pandas 完全一致。这正是 dask.dataframe 底层做的事。
# English: a "big" DataFrame split into partitions (each a plain pandas chunk). groupby-sum = per-partition groupby
#      then combine, identical to single-machine pandas. This IS what dask.dataframe does underneath.
# ============================================================
import pandas as pd, numpy as np
np.random.seed(0)
big=pd.DataFrame({"key":np.random.choice(list("ABCD"),200000),
                  "value":np.random.randint(1,100,200000)})
# 单机 pandas 基线 / single-machine baseline
single=big.groupby("key")["value"].sum()
# "Dask" 方式:切 8 个分区, 每个分区各自 groupby-sum, 再把部分和相加 / partitioned + combine
parts=[big.iloc[ix] for ix in np.array_split(np.arange(len(big)), 8)] # 8 个分区(普通 pandas 块)/ 8 pandas chunks
partials=[p.groupby("key")["value"].sum() for p in parts]            # 每分区局部 groupby(可并行)/ per-partition (parallelizable)
combined=pd.concat(partials).groupby(level=0).sum()                  # 合并部分结果 / combine partial results
print("单机 pandas groupby-sum / single-machine:\n", single.to_dict())
print("分区(Dask 方式)groupby-sum / partitioned:\n", combined.to_dict())
print("完全一致 / identical:", (single.sort_index()==combined.sort_index()).all())
print("→ groupby-sum 是可分解的(部分和可再合并), 所以能分区并行——和 21.3 梯度可加同一个道理")


In [ ]:

# ============================================================
# 可视化:任务图 DAG + 并行加速 / task graph DAG + parallel speedup
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5.5))
# ① 任务图(手绘 3 分支合并结构)/ task graph (3 branches merging)
ax[0].axis("off"); ax[0].set_title("Dask 任务图:3 条独立分支并行 → 合并",fontsize=12,weight="bold")
for i,x0 in enumerate([0.12,0.45,0.78]):
    ys=[0.8,0.6,0.4]; labels=["load","square","total"]; col=["#DD8452","#4C72B0","#55A868"]
    for j,(y,lab,c) in enumerate(zip(ys,labels,col)):
        ax[0].add_patch(plt.Circle((x0,y),0.05,fc=c,alpha=0.8,transform=ax[0].transAxes))
        ax[0].text(x0,y,lab,ha="center",va="center",fontsize=7,color="white",transform=ax[0].transAxes)
        if j>0: ax[0].annotate("",xy=(x0,ys[j]+0.05),xytext=(x0,ys[j-1]-0.05),arrowprops=dict(arrowstyle="->"),transform=ax[0].transAxes)
    ax[0].annotate("",xy=(0.5,0.2),xytext=(x0,0.35),arrowprops=dict(arrowstyle="->",color="gray"),transform=ax[0].transAxes)
ax[0].add_patch(plt.Circle((0.5,0.15),0.055,fc="#C44E52",alpha=0.8,transform=ax[0].transAxes))
ax[0].text(0.5,0.15,"combine",ha="center",va="center",fontsize=7,color="white",transform=ax[0].transAxes)
ax[0].text(0.5,0.02,"独立分支(不同颜色列)可并行执行 / independent branches run in parallel",ha="center",fontsize=8,style="italic",transform=ax[0].transAxes)
# ② 加速 / speedup
ax[1].bar(["串行\nsequential","并行\nparallel"],[t_seq,t_par],color=["#C44E52","#55A868"])
for i,v in enumerate([t_seq,t_par]): ax[1].text(i,v+0.02,f"{v:.2f}s",ha="center",fontsize=11,weight="bold")
ax[1].set_ylabel("耗时 seconds"); ax[1].set_title(f"独立任务并行 → {t_seq/t_par:.1f}x 加速")
plt.tight_layout(); plt.savefig("/tmp/big05_viz.png",dpi=80); plt.show()
print("左:计算被织成任务图, 独立分支并行; 右:并行调度带来接近分支数的加速")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **任务图是"并行"的通用语言**:Dask 的全部魔力,就是把你写的普通 Python 计算**自动拆成一张 DAG**,然后让调度器把没有依赖关系的节点并行跑。我们几十行的 mini-Dask 就拿到了 3 倍加速(3 条独立分支同时算)。这和 Spark 的 DAG、我们前面写的 MiniSpark,是**同一个思想的不同外壳**——惰性构图、找出独立部分、并行执行。区别在于:Spark 图的节点是"分区上的 RDD 算子",Dask 图的节点是"任意 Python 函数",所以 **Dask 能并行化连 Spark 都表达不了的自定义 Python 逻辑**(科学计算、自定义模拟、任意库调用)。
2. **`dask.dataframe` 只是"分区的 pandas + 可分解的聚合"**:我们看到 `groupby-sum` 能被拆成"每个分区各自 groupby,再把部分和合并",结果和单机 pandas 一模一样。这和 21.3 的"梯度可加"是**完全一样的数学结构**:只要一个操作满足"局部算 + 合并 = 全局算"(可结合的聚合),它就能分区并行。这也解释了 Dask/Spark 的共同软肋——**不满足这个结构的操作(如需要全局排序、跨分区去重、复杂 join)必须 shuffle,慢且贵**。
3. **诚实的边界:Dask 常常不是最快的,选型要清醒**。①**小数据上,任务图调度本身有开销**——构图、序列化、线程/进程协调,当数据能轻松进内存时,**纯 pandas 往往比 Dask 快**(Dask 的价值在数据超内存或计算能大规模并行时才显现)。②**单机极速另有王者**:如果只是单机上想更快地处理几十 GB,**Polars / DuckDB(C++/Rust 引擎,列式、向量化)通常比 Dask 快得多也简单**(下两节实测)。Dask 的真正甜区是:**你已经有一大堆 pandas/numpy/sklearn 代码,想几乎不改代码地扩展到多核甚至集群**,或者要编排**自定义的并行 Python 工作流**。③**别神化"分布式"**:和前几节一样,先诚实评估数据规模和计算模式,再选工具——Dask、Spark、Polars、DuckDB、甚至纯 pandas,各有各的甜区,**用错工具比不会用工具代价更大**。

**English**:
1. **The task graph is the universal language of "parallel"**: all of Dask's magic is **automatically breaking your ordinary Python computation into a DAG**, then letting the scheduler run dependency-free nodes in parallel. Our few-dozen-line mini-Dask already gets a 3x speedup (3 independent branches computed at once). This is the **same idea in a different shell** as Spark's DAG and our earlier MiniSpark — build lazily, find independent parts, execute in parallel. The difference: Spark's graph nodes are "RDD operators on partitions," while Dask's are "arbitrary Python functions," so **Dask can parallelize custom Python logic even Spark can't express** (scientific computing, custom simulations, arbitrary library calls).
2. **`dask.dataframe` is just "partitioned pandas + decomposable aggregation"**: we saw `groupby-sum` split into "each partition groupby, then combine partial sums," identical to single-machine pandas. This is the **exact same mathematical structure** as 21.3's "additive gradients": as long as an operation satisfies "local compute + combine = global compute" (an associative aggregation), it can be partition-parallelized. This also explains Dask/Spark's common weakness — **operations that don't satisfy this structure (needing global sorts, cross-partition deduplication, complex joins) must shuffle, slow and costly**.
3. **Honest limits: Dask is often not the fastest — choose tools soberly**. ① **On small data, the task-graph scheduling itself has overhead** — graph building, serialization, thread/process coordination; when data fits easily in memory, **plain pandas is often faster than Dask** (Dask's value appears only when data exceeds memory or computation parallelizes at scale). ② **For single-machine speed there are other kings**: to just process tens of GB faster on one machine, **Polars / DuckDB (C++/Rust engines, columnar, vectorized) are usually far faster and simpler than Dask** (benchmarked in the next two sections). Dask's true sweet spot is: **you already have a pile of pandas/numpy/sklearn code and want to scale it to cores or a cluster with almost no code changes**, or to orchestrate **custom parallel Python workflows**. ③ **Don't mythologize "distributed"**: as in prior sections, honestly assess data size and computation pattern first, then pick a tool — Dask, Spark, Polars, DuckDB, even plain pandas each have a sweet spot, and **using the wrong tool costs more than not mastering a tool**.

> 💼 **实战视角 / Practical angle**
> **中文**:Dask 落地:①**扩展现有 Python**——把 pandas 换成 `dask.dataframe`、numpy 换 `dask.array`、加 `@dask.delayed` 并行化自定义函数, 几乎不改逻辑;②**out-of-core**(数据略超内存)——Dask 分区流式处理, 不用一次性加载;③`dask.distributed` 起集群 + 自带 dashboard 看任务图/进度/内存;④`dask-ml` 做并行/大数据机器学习。**选型速记**:能进内存的小数据→**pandas**; 单机想更快→**Polars/DuckDB**; 有大量现成 Python 代码要扩到多核/多机、或 out-of-core→**Dask**; JVM 生态/超大规模/成熟容错→**Spark**。**坑**:别对小数据上 Dask(调度开销); 减少 shuffle(groupby/join 跨分区)。面试金句:*"Dask 用惰性任务图并行化任意 Python——delayed 织 DAG、调度器跑独立任务、dask.dataframe 是分区 pandas 且聚合可分解; 甜区是扩展现有 pandas/numpy 代码到多核多机或 out-of-core; 但小数据用 pandas、单机极速用 Polars/DuckDB、超大规模用 Spark——按数据规模和计算模式选工具。"*
> **English**: Dask in practice: ① **scale existing Python** — swap pandas for `dask.dataframe`, numpy for `dask.array`, add `@dask.delayed` to parallelize custom functions, with almost no logic change; ② **out-of-core** (data slightly over memory) — Dask streams partitions without loading all at once; ③ `dask.distributed` starts a cluster + built-in dashboard for the task graph/progress/memory; ④ `dask-ml` for parallel/big-data ML. **Tool cheat-sheet**: small data that fits memory → **pandas**; want single-machine speed → **Polars/DuckDB**; lots of existing Python to scale to cores/machines or out-of-core → **Dask**; JVM ecosystem/huge scale/mature fault tolerance → **Spark**. **Pitfalls**: don't Dask small data (scheduling overhead); minimize shuffle (cross-partition groupby/join). Interview line: *"Dask parallelizes arbitrary Python via a lazy task graph — delayed weaves a DAG, the scheduler runs independent tasks, dask.dataframe is partitioned pandas with decomposable aggregation; its sweet spot is scaling existing pandas/numpy code to cores/machines or out-of-core; but use pandas for small data, Polars/DuckDB for single-machine speed, Spark for huge scale — pick the tool by data size and computation pattern."*

---
### 小结 / Summary
- **中文**:Dask=纯 Python 并行, 核心是惰性任务图(DAG)+ 调度器并行跑独立任务; dask.dataframe=分区 pandas。
- **English**: Dask = pure-Python parallelism; core is a lazy task graph (DAG) + scheduler running independent tasks in parallel; dask.dataframe = partitioned pandas.
- **中文**:可分解聚合(局部算+合并=全局)能分区并行(同"梯度可加"); 不满足的操作要 shuffle, 慢。
- **English**: Decomposable aggregations (local + combine = global) partition-parallelize (like "additive gradients"); others need shuffle, slow.
- **中文**:选型:小数据 pandas、单机极速 Polars/DuckDB、扩展现有 Python/out-of-core 用 Dask、超大规模用 Spark。
- **English**: Tool choice: pandas for small data, Polars/DuckDB for single-machine speed, Dask to scale existing Python/out-of-core, Spark for huge scale.
